In [ ]:
from description_prompts import *
from openai import OpenAI
from collections import defaultdict
import os
import json
import pandas as pd
import pickle
from prompts import *

API_key = ""
path_to_nist_assessments = "iKAT2025/interactive/runs/trec-ikat25-nist-assessments/"
path_to_output_file = "result.pkl"


client = OpenAI(api_key=API_key)


def run_api(prompt):
    response = client.responses.create(
        model="gpt-4.1-mini",  # or "gpt-4.1", "gpt-4o-mini", etc.
        input=prompt,
        max_output_tokens=50
    )

    return response.output_text

def run_api_gpt5(prompt):
    response = client.responses.create(
        model="gpt-5",  # or "gpt-4.1", "gpt-4o-mini", "gpt-5-mini" etc. 
        input=prompt,
        reasoning={"effort": "medium"},
        text={"verbosity": "low"},
        # max_output_tokens=200
        # max_output_tokens=500
    )
    # print(response)

    return response.output_text

def get_judgment_type(judgment):
    if len(judgment.keys()) == 5:
        return 'rubric'
    else:
        return 'dialog'

def eval_rubric(row):
    all_ptkb = []
    dialog = ""
    for turn_data in row['responses']:
        all_ptkb += turn_data['response']['ptkb_provenance']
        dialog += f"user: {turn_data['response']['user_utterance']}\n"
        text = turn_data['response'].get('text') or ""
        response = text.replace('\n', ' ')
        dialog += f"system: {response}\n"

    ptkb = '\n'.join(list(set(all_ptkb)))
    prompt = rubric_level_prompt.format(PTKB = ptkb, history = dialog, engagement = engagement,  relevance_and_usefulness=relevance_and_usefulness,  overall_subtopic_quality = overall_subtopic_quality, rater_confidence=rater_confidence)
    print(prompt)

    return prompt

def eval_rubric_single_metric(row):
    all_ptkb = []
    prompt_arr = []

    dialog = ""
    for turn_data in row['responses']:
        all_ptkb += turn_data['response']['ptkb_provenance']
        dialog += f"user: {turn_data['response']['user_utterance']}\n"
        text = turn_data['response'].get('text') or ""
        response = text.replace('\n', ' ')
        dialog += f"system: {response}\n"

    ptkb = '\n'.join(list(set(all_ptkb)))

    prompt_arr.append(rubric_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_description = engagement, aspect_name = 'engagement'))
    prompt_arr.append(rubric_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_description = relevance_and_usefulness, aspect_name = 'relevance_and_usefulness'))
    prompt_arr.append(rubric_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_description = overall_subtopic_quality, aspect_name = 'overall_subtopic_quality'))


    return prompt_arr

def eval_rubric_single_metric_short(row):
    all_ptkb = []
    prompt_arr = []

    dialog = ""
    for turn_data in row['responses']:
        all_ptkb += turn_data['response']['ptkb_provenance']
        dialog += f"user: {turn_data['response']['user_utterance']}\n"
        text = turn_data['response'].get('text') or ""
        response = text.replace('\n', ' ')
        dialog += f"system: {response}\n"

    ptkb = '\n'.join(list(set(all_ptkb)))

    prompt_arr.append(rubric_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_description = engagement_short, aspect_name = 'engagement'))
    prompt_arr.append(rubric_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_description = relevance_and_usefulnesst_short, aspect_name = 'relevance_and_usefulness'))
    prompt_arr.append(rubric_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_description = overall_subtopic_qualityt_short, aspect_name = 'overall_subtopic_quality'))


    return prompt_arr

def eval_dialog(row):
    all_ptkb = []
    dialog = ""
    for turn_data in row['responses']:
        all_ptkb += turn_data['response']['ptkb_provenance']
        dialog += f"user: {turn_data['response']['user_utterance']}\n"
        text = turn_data['response'].get('text') or ""
        response = text.replace('\n', ' ')
        dialog += f"system: {response}\n"

    ptkb = '\n'.join(list(set(all_ptkb)))
    prompt = dialog_level_prompt.format(PTKB = ptkb, history = dialog, mixed_initiative_strategies = mixed_initiative_strategies, personalization = personalization, information_flow_and_coherence = information_flow_and_coherence, trustworthiness = trustworthiness, rater_confidence = rater_confidence, overall_user_satisfaction=overall_user_satisfaction)
    print(prompt)
    # result = ''
    # result = run_api(prompt)

    return prompt

def eval_dialog_single(row):
    all_ptkb = []
    prompt_arr = []
    dialog = ""

    for turn_data in row['responses']:
        all_ptkb += turn_data['response']['ptkb_provenance']
        dialog += f"user: {turn_data['response']['user_utterance']}\n"
        text = turn_data['response'].get('text') or ""
        response = text.replace('\n', ' ')
        dialog += f"system: {response}\n"

    ptkb = '\n'.join(list(set(all_ptkb)))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='mixed_initiative_strategies' , aspect_description=mixed_initiative_strategies))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='personalization' , aspect_description=personalization))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='information_flow_and_coherence' , aspect_description=information_flow_and_coherence))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='trustworthiness' , aspect_description=trustworthiness))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='overall_user_satisfaction' , aspect_description=overall_user_satisfaction))


    return prompt_arr

def eval_dialog_single_short(row):
    all_ptkb = []
    prompt_arr = []
    dialog = ""

    for turn_data in row['responses']:
        all_ptkb += turn_data['response']['ptkb_provenance']
        dialog += f"user: {turn_data['response']['user_utterance']}\n"
        text = turn_data['response'].get('text') or ""
        response = text.replace('\n', ' ')
        dialog += f"system: {response}\n"

    ptkb = '\n'.join(list(set(all_ptkb)))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='mixed_initiative_strategies' , aspect_description=mixed_initiative_strategiest_short))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='personalization' , aspect_description=personalizationt_short))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='information_flow_and_coherence' , aspect_description=information_flow_and_coherencet_short))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='trustworthiness' , aspect_description=trustworthinesst_short))
    prompt_arr.append(dialog_level_single_prompt.format(PTKB = ptkb, history = dialog, aspect_name='overall_user_satisfaction' , aspect_description=overall_user_satisfactiont_short))


    return prompt_arr




In [ ]:




results_file_names = os.listdir(path_to_nist_assessments)
all_samples = []

for file_name in results_file_names:
    if file_name.endswith('.final'):
        path_to_result = f"{path_to_nist_assessments}{file_name}"
        print(path_to_result)

        with open(path_to_result, 'r') as f:
            data = json.load(f)
        

        index = 1
        for row in data:
            run_name = row['runtag']
            topic_id = row['topic']
            type_ = row['type']
            judgement_type = get_judgment_type(row['judgment'])
            
            if judgement_type == 'dialog':
                prompts = eval_dialog_single(row)
                for result in prompts:
                    all_samples.append({"run_name": run_name,
                                        "topic_id": topic_id,
                                        "type_": type_,
                                        "index": index,
                                        "nist-judgment":row['judgment'],
                                        "judgement_type": judgement_type,
                                        "prompt": result})
                
            elif judgement_type == 'rubric':
                prompts = eval_rubric_single_metric(row)
                for result in prompts:
                    all_samples.append({"run_name": run_name,
                                        "topic_id": topic_id,
                                        "type_": type_,
                                        "index": index,
                                        "nist_judgment":row['judgment'],
                                        "judgement_type": judgement_type,
                                        "prompt": result})
            
            index += 1


print(len(all_samples))



with open(path_to_output_file, 'wb') as f:
    pickle.dump(all_samples, f)

print(all_samples[100]['prompt'])

In [ ]:
with open(path_to_output_file, 'rb') as f:
    all_samples = pickle.load(f)

index = 0
for sample in all_samples:
    index += 1
    if 'llm-judgment' in sample:
        continue
    
    sample['llm-judgment'] = run_api_gpt5(sample['prompt'])
    # sample['llm-judgment'] = run_api(sample['prompt'])
    # print(sample['llm-judgment'])

    if index%20 == 1:
        with open(path_to_output_file, 'wb') as f:
            pickle.dump(all_samples, f)
        print(sample['llm-judgment'])
        print(f'******\nSaved at step: {index}\n*********')


with open(path_to_output_file, 'wb') as f:
    pickle.dump(all_samples, f)
